# 01 — Pretrain LocalMind

Thin launcher for the SS8 Phase 4 pretraining run on 2x T4.

**Budget:** ~7 h for the 1.5B-token 31M run; 30 GPU-h/week quota; 12 h hard session cap.
**Before running:** set `REPO` below and add `HF_TOKEN` to Kaggle Secrets.


In [ ]:
# Thin launcher: clone, install, run. No project logic lives in this notebook.
import subprocess, sys, os, pathlib
REPO = 'https://github.com/adnanbasil10/localmind.git'
WORK = pathlib.Path('/kaggle/working/localmind')
if not WORK.exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(WORK)], check=True)
os.chdir(WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[torch,tok,data]'], check=True)
import torch; print('torch',torch.__version__,'| cuda',torch.cuda.is_available(),
      '|', torch.cuda.device_count(),'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Hardware pre-flight. T4 is SM 7.5 (Turing): it has NO bf16 tensor cores, so this project
# mandates fp16 + GradScaler everywhere (see docs/decisions/0001-fp16-gradscaler-not-bf16.md).
#
# NOTE: torch.cuda.is_bf16_supported() returns True even on a T4, because PyTorch reports
# EMULATED bf16 support. Emulated bf16 is slow and defeats the purpose, so it is not a signal
# worth asserting on. The real invariant is that our config asks for fp16.
import torch, yaml

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU            : {torch.cuda.get_device_name(0)}  x{torch.cuda.device_count()}')
    print(f'compute cap    : {cap}  ->', 'Turing/pre-Ampere' if cap[0] < 8 else 'Ampere+')
    print(f'bf16 (emulated): {torch.cuda.is_bf16_supported()}  <- ignore; no bf16 tensor cores below SM 8.0')
else:
    print('No GPU. Set Accelerator = GPU T4 x2 in the right sidebar.')

# The invariant that actually matters: never train this project in bf16.
cfg = yaml.safe_load(open('configs/train/pretrain.yaml'))
assert cfg['precision'] == 'fp16', f"config must be fp16, got {cfg['precision']!r}"
print('config precision:', cfg['precision'], '-- correct for this hardware')


In [ ]:
# Session-cap insurance (SS3.2 item 3): push checkpoints to HF Hub every hour.
# The HF username is derived from your token, so there is nothing to edit here.
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import whoami
user = whoami(token=os.environ['HF_TOKEN'])['name']
os.environ['LOCALMIND_HUB_REPO'] = f'{user}/localmind-31m'
print('checkpoints will push to:', os.environ['LOCALMIND_HUB_REPO'])


## 1. Smoke test — proves the loop works, no data needed

`--synthetic` trains against a deterministic in-memory corpus. Watch `ce_loss` fall.
Takes ~2 minutes. **Do not skip this** — it costs almost nothing and catches setup bugs
before you spend 7 GPU-hours.


In [ ]:
!python -W ignore::RuntimeWarning -m localmind.train.loop --config configs/train/smoke.yaml --synthetic --max-steps 50 --resume none


## 2. Build the real training data

Streams the mixture from Hugging Face, cleans it, deduplicates it, tokenizes it, and
writes `uint16` shards. Trains the tokenizer first if one does not exist.

Start with `--n-docs 20000` to confirm the pipeline end to end (~10 min), then raise it
for the real run. `--smoke` uses a tiny built-in corpus and no network at all.


In [ ]:
# Quick end-to-end check of the data pipeline (no network):
!python -W ignore::RuntimeWarning -m localmind.data.prepare --smoke --out /kaggle/working/shards_smoke --tokenizer /kaggle/working/tok_smoke.json --vocab-size 2048

# The real thing (streams from Hugging Face; needs Internet = On):
# !python -W ignore::RuntimeWarning -m localmind.data.prepare --n-docs 20000 --out /kaggle/working/shards --tokenizer /kaggle/working/tokenizer.json


## 3. Main pretrain (2x T4)

Point the loop at the shards you just built. Resumes bit-exactly if the session dies —
just re-run this cell.


In [ ]:
import os
os.environ['LOCALMIND_SHARD_DIR'] = '/kaggle/working/shards'

!torchrun --nproc_per_node=2 -m localmind.train.loop --config configs/train/pretrain.yaml --resume auto
